# DeFi Data Pipeline — FinTech 590
Fetches top Uniswap V3 pools, verifies contracts on Sourcify, and saves results.

## 0. Install Dependencies

In [1]:
import subprocess, sys

def ensure(*packages):
    for pkg in packages:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            print(f"[setup] Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

ensure("requests", "pandas", "python-dotenv", "web3")
print("All dependencies ready.")

[setup] Installing python-dotenv...
All dependencies ready.


## 1. Config

In [2]:
import os, json, time, pathlib
import requests
import pandas as pd
from dotenv import load_dotenv
from web3 import Web3

load_dotenv()

DEFILLAMA_POOLS = "https://yields.llama.fi/pools"
SOURCIFY_CHECK  = "https://sourcify.dev/server/v2/contract/1/"
SOURCIFY_FILES  = "https://sourcify.dev/server/files/any/1/"
RATE_LIMIT_DELAY = 0.25
MIN_TVL_USD      = 1_000_000
TOP_N            = 20

# Uniswap V3 factory constants (Ethereum mainnet)
UNISWAP_V3_FACTORY  = "0x1F98431c8aD98523631AE4a59f267346ea31F984"
POOL_INIT_CODE_HASH = "e34f199b19b2b4f47f68442619d555527d244f78a3297ea89325f843f87b8b54"

ABI_DIR  = pathlib.Path("pool_abis")
CSV_PATH = pathlib.Path("top_pools.csv")
ABI_DIR.mkdir(exist_ok=True)

print(f"Output: {CSV_PATH}  |  ABIs: {ABI_DIR}/")
print("Contract verification: Sourcify (no API key required)")

Output: top_pools.csv  |  ABIs: pool_abis/
Contract verification: Sourcify (no API key required)


## 2. Fetch Top Uniswap V3 Pools from DeFiLlama

> DeFiLlama returns internal UUIDs rather than on-chain addresses, so we compute the real
> pool address from token addresses + fee tier using Uniswap V3's deterministic CREATE2 formula.

In [3]:
def compute_pool_address(token_a: str, token_b: str, fee: int) -> str:
    """
    Derive the Uniswap V3 pool address from its two tokens and fee tier
    using the same CREATE2 formula as the factory contract.
    """
    token0, token1 = sorted([token_a.lower(), token_b.lower()])
    encoded = (
        bytes.fromhex(token0[2:]).rjust(32, b'\x00') +
        bytes.fromhex(token1[2:]).rjust(32, b'\x00') +
        fee.to_bytes(32, 'big')
    )
    salt = Web3.keccak(encoded)
    data = (
        b'\xff' +
        bytes.fromhex(UNISWAP_V3_FACTORY[2:]) +
        salt +
        bytes.fromhex(POOL_INIT_CODE_HASH)
    )
    return Web3.to_checksum_address('0x' + Web3.keccak(data)[12:].hex())


def parse_fee_tier(pool_meta: str | None) -> int:
    """Convert poolMeta string like '0.05%' to Uniswap fee pips (500)."""
    try:
        return int(float(pool_meta.replace("%", "")) * 10_000)
    except (AttributeError, ValueError):
        return 0


print("Fetching pool data from DeFiLlama...")
resp = requests.get(DEFILLAMA_POOLS, timeout=30)
resp.raise_for_status()
all_pools = resp.json()["data"]
print(f"  Total pools across all protocols: {len(all_pools):,}")

uni_pools = [
    p for p in all_pools
    if p.get("project") == "uniswap-v3"
    and p.get("chain") == "Ethereum"
    and (p.get("tvlUsd") or 0) >= MIN_TVL_USD
]
uni_pools.sort(key=lambda x: x.get("tvlUsd", 0), reverse=True)
top_raw = uni_pools[:TOP_N]
print(f"  Uniswap V3 / Ethereum / TVL>${MIN_TVL_USD/1e6:.0f}M: {len(uni_pools)} found, keeping top {len(top_raw)}\n")

pools = []
for p in top_raw:
    underlying    = p.get("underlyingTokens") or []
    fee           = parse_fee_tier(p.get("poolMeta"))
    symbol_parts  = p.get("symbol", "-").split("-")
    address = (
        compute_pool_address(underlying[0], underlying[1], fee)
        if len(underlying) >= 2 and fee > 0
        else p["pool"]
    )
    pools.append({
        "address":    address,
        "token0":     symbol_parts[0] if len(symbol_parts) > 0 else "",
        "token1":     symbol_parts[1] if len(symbol_parts) > 1 else "",
        "fee_tier":   fee,
        "tvl_usd":    float(p.get("tvlUsd") or 0),
        "volume_usd": float(p.get("volumeUsd1d") or 0),
    })

for i, p in enumerate(pools, 1):
    print(f"  {i:>2}. {p['token0']}/{p['token1']} "
          f"(fee {p['fee_tier']/1e4:.2f}%)  "
          f"TVL=${p['tvl_usd']:>14,.0f}  "
          f"addr={p['address'][:10]}...")

Fetching pool data from DeFiLlama...
  Total pools across all protocols: 18,167
  Uniswap V3 / Ethereum / TVL>$1M: 134 found, keeping top 20

   1. USDC/WETH (fee 0.05%)  TVL=$    95,094,285  addr=0x88e6A0c2...
   2. WETH/USDT (fee 0.30%)  TVL=$    63,896,883  addr=0x4e68Ccd3...
   3. WBTC/WETH (fee 0.05%)  TVL=$    45,551,841  addr=0x4585FE77...
   4. WBTC/WETH (fee 0.30%)  TVL=$    44,526,108  addr=0xCBCdF962...
   5. WBTC/CBBTC (fee 0.01%)  TVL=$    29,428,811  addr=0xe8f7c89C...
   6. WBTC/USDC (fee 0.30%)  TVL=$    27,036,724  addr=0x99ac8cA7...
   7. USDC/USDT (fee 0.01%)  TVL=$    26,343,255  addr=0x3416cF6C...
   8. WBTC/USDT (fee 0.30%)  TVL=$    26,288,934  addr=0x9Db9e0e5...
   9. AUSD/USDC (fee 0.01%)  TVL=$    25,010,945  addr=0xbAFeAd7c...
  10. USDC/WETH (fee 0.30%)  TVL=$    22,298,240  addr=0x8ad599c3...
  11. LINK/WETH (fee 0.30%)  TVL=$    22,140,600  addr=0xa6Cc3C25...
  12. USDM/USDT (fee 0.05%)  TVL=$    19,825,137  addr=0x4a25dBDF...
  13. WM/USDC (fee 0.01%)  TV

## 3. Verify Each Pool Contract on Sourcify

> [Sourcify](https://sourcify.dev) is a free, open-source smart contract verification service.  
> No API key required. Uniswap V3 pools are verified on Sourcify via the Uniswap team.

In [4]:
def sourcify_check(address: str) -> tuple[bool, list | None]:
    """
    Check if a contract is verified on Sourcify and return its ABI.
    Returns (is_verified, abi_or_None).
    """
    try:
        # Step 1: check verification status
        r = requests.get(SOURCIFY_CHECK + address, timeout=15)
        if r.status_code == 404:
            return False, None
        r.raise_for_status()
        match = r.json().get("match")  # 'match', 'partial', or absent
        if not match:
            return False, None

        # Step 2: fetch source files and extract ABI from metadata.json
        r2 = requests.get(SOURCIFY_FILES + address, timeout=15)
        r2.raise_for_status()
        files = r2.json().get("files", [])
        meta  = next((f for f in files if f["name"] == "metadata.json"), None)
        if meta:
            abi = json.loads(meta["content"])["output"]["abi"]
            return True, abi
        return True, None  # verified but couldn't get ABI

    except requests.RequestException:
        return False, None


for p in pools:
    addr = p["address"]
    print(f"  Checking {addr[:10]}...  ({p['token0']}/{p['token1']})", end="", flush=True)
    verified, abi = sourcify_check(addr)
    p["etherscan_verified"] = verified

    if verified and abi:
        abi_path = ABI_DIR / f"{addr}.json"
        abi_path.write_text(json.dumps(abi, indent=2))
        print(f"  ✓ verified  ({len(abi)} ABI entries saved → pool_abis/{addr}.json)")
    elif verified:
        print(f"  ✓ verified  (ABI unavailable)")
    else:
        print(f"  ✗ not verified")

    time.sleep(RATE_LIMIT_DELAY)

  Checking 0x88e6A0c2...  (USDC/WETH)  ✓ verified  (36 ABI entries saved → pool_abis/0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640.json)
  Checking 0x4e68Ccd3...  (WETH/USDT)  ✓ verified  (36 ABI entries saved → pool_abis/0x4e68Ccd3E89f51C3074ca5072bbAC773960dFa36.json)
  Checking 0x4585FE77...  (WBTC/WETH)  ✓ verified  (36 ABI entries saved → pool_abis/0x4585FE77225b41b697C938B018E2Ac67Ac5a20c0.json)
  Checking 0xCBCdF962...  (WBTC/WETH)  ✓ verified  (36 ABI entries saved → pool_abis/0xCBCdF9626bC03E24f779434178A73a0B4bad62eD.json)
  Checking 0xe8f7c89C...  (WBTC/CBBTC)  ✓ verified  (36 ABI entries saved → pool_abis/0xe8f7c89C5eFa061e340f2d2F206EC78FD8f7e124.json)
  Checking 0x99ac8cA7...  (WBTC/USDC)  ✓ verified  (36 ABI entries saved → pool_abis/0x99ac8cA7087fA4A2A1FB6357269965A2014ABc35.json)
  Checking 0x3416cF6C...  (USDC/USDT)  ✓ verified  (36 ABI entries saved → pool_abis/0x3416cF6C708Da44DB2624D63ea0AAef7113527C6.json)
  Checking 0x9Db9e0e5...  (WBTC/USDT)  ✓ verified  (36 ABI en

## 4. Save Results

In [5]:
df = pd.DataFrame(pools, columns=[
    "address", "token0", "token1", "fee_tier",
    "tvl_usd", "volume_usd", "etherscan_verified"
])
df.to_csv(CSV_PATH, index=False)

verified_count = df["etherscan_verified"].sum()
abi_files      = list(ABI_DIR.glob("*.json"))

print(f"top_pools.csv  → {len(df)} rows saved")
print(f"pool_abis/     → {len(abi_files)} ABI files ({verified_count} verified pools)")
print()
df[["token0", "token1", "fee_tier", "tvl_usd", "etherscan_verified"]]

top_pools.csv  → 20 rows saved
pool_abis/     → 19 ABI files (19 verified pools)



,token0,token1,fee_tier,tvl_usd,etherscan_verified
0,USDC,WETH,500,95094285.0,True
1,WETH,USDT,3000,63896883.0,True
2,WBTC,WETH,500,45551841.0,True
3,WBTC,WETH,3000,44526108.0,True
4,WBTC,CBBTC,100,29428811.0,True
5,WBTC,USDC,3000,27036724.0,True
6,USDC,USDT,100,26343255.0,True
7,WBTC,USDT,3000,26288934.0,True
8,AUSD,USDC,100,25010945.0,True
9,USDC,WETH,3000,22298240.0,True
